# Notebook 03 - Calibration Model Development

**Milestone:** HV-006  
**Technical Gate:** CM-001  
**Primary Engineering Question:** Calibration Model Development

This notebook develops and freezes an RH calibration model using only qualified analyst-visible `STATIC / CALIBRATION` evidence from the frozen `VSM-LAB-DATA-v1.1` release.

The notebook must preserve the analyst-blind boundary established by DQ-001 and the calibration/validation separation required by repository governance.

**No VALIDATION measurement result is used for fitting, tuning, feature selection, or model selection in HV-006.**

## 1. Engineering Question

Can a calibration model built only from the frozen CALIBRATION subset reduce systematic RH measurement error while remaining:

- interpretable;
- free from experimental-unit leakage;
- operationally deployable; and
- suitable for later blind independent validation?

HV-006 answers the **model-development** question only.

The independent question—

> Does the frozen calibration generalize to reserved validation data?

—is deferred to Notebook 04.

## 2. Scientific Boundary

### Included

- `STATIC / CALIBRATION` measurement records only;
- public test-matrix design metadata;
- Test_ID-level calibration summaries;
- DUT-observable candidate predictors;
- calibration-only internal cross-validation;
- calibration-only residual diagnostics;
- final model freeze.

### Prohibited in HV-006

- VALIDATION measurement values or validation performance metrics;
- hidden simulator truth;
- private DUT personality/configuration;
- target-side/reference variables as deployable predictors;
- sample-level random train/test splitting;
- model choice informed by Notebook 04.

The public test matrix may be used to verify that a reserved validation design exists, but its measurement values remain outside the HV-006 evidence boundary.

## 3. Experimental Unit and Calibration Target

The primary model-development unit is **one summary record per CALIBRATION Test_ID**.

Each Test_ID contains 60 repeated 1 Hz measurements under one nominal static condition. Those 60 samples are useful for estimating the Test_ID mean and within-test scatter, but they are **not 60 independent calibration experiments**.

Therefore:

```text
5,760 CALIBRATION samples
        ↓ summarize by Test_ID
96 calibration modeling records
```

The pre-calibration error convention remains:

```text
Error_RH_Pre = RH_DUT_Mean - RH_Reference_Mean
```

The correction target is:

```text
Correction_RH = RH_Reference_Mean - RH_DUT_Mean
              = -Error_RH_Pre
```

Later, the calibrated result will be:

```text
RH_Calibrated = RH_DUT + Predicted_Correction_RH
```

and the post-calibration residual will use:

```text
Residual_Cal = RH_Calibrated - RH_Reference
```

The reference indication is a reference measurement, not exact physical truth.

## 4. Reproducible Setup

Reusable calibration loading and Test_ID-summary logic is kept in `src/calibration_analysis.py`.

This notebook remains the engineering record: it verifies the analysis boundary, shows the design, constructs the modeling table, and later documents model comparison and freeze decisions.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold


# ------------------------------------------------------------
# Locate project root and import reusable analyst-side helpers
# ------------------------------------------------------------

current_dir = Path.cwd()
candidate_roots = [current_dir, *current_dir.parents]

project_root = next(
    (
        path
        for path in candidate_roots
        if (path / "data" / "raw" / "vsm_lab_data_v1_1").exists()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root containing "
        "'data/raw/vsm_lab_data_v1_1'."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.calibration_analysis import (
    calibration_design_summary,
    load_static_calibration,
    summarize_calibration_test_ids,
)

reports_dir = project_root / "reports"
figures_dir = reports_dir / "figures"
tables_dir = reports_dir / "tables"

figures_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root : {project_root}")
print(f"Figures dir : {figures_dir}")
print(f"Tables dir  : {tables_dir}")

## 5. CM-01 — CALIBRATION Data Boundary Audit

The first executable requirement is to prove that the working measurement frame contains only `STATIC / CALIBRATION` evidence.

Expected calibration design from the frozen public test matrix:

```text
4 RH levels × 3 temperatures × 2 directions × 2 days × 2 runs
= 96 CALIBRATION Test_IDs

96 Test_IDs × 60 samples/Test_ID
= 5,760 measurement rows
```

The reserved validation design is checked only through the public test matrix. No validation measurement values are summarized or evaluated here.

In [ ]:
# ------------------------------------------------------------
# CM-01A: Load calibration evidence and public design metadata
# ------------------------------------------------------------

calibration_df, test_matrix_df = load_static_calibration(project_root)

calibration_matrix_df = test_matrix_df.loc[
    (test_matrix_df["Test_Family"] == "STATIC")
    & (test_matrix_df["Analysis_Role"] == "CALIBRATION")
].copy()

validation_matrix_df = test_matrix_df.loc[
    (test_matrix_df["Test_Family"] == "STATIC")
    & (test_matrix_df["Analysis_Role"] == "VALIDATION")
].copy()

samples_per_test = calibration_df.groupby("Test_ID").size()

print(f"CALIBRATION measurement rows : {len(calibration_df):,}")
print(f"CALIBRATION Test_IDs          : {calibration_df['Test_ID'].nunique()}")
print(f"CALIBRATION matrix Test_IDs   : {calibration_matrix_df['Test_ID'].nunique()}")
print(f"Reserved VALIDATION Test_IDs  : {validation_matrix_df['Test_ID'].nunique()}")
print(
    "Samples/Test_ID values       : "
    f"{sorted(samples_per_test.unique().tolist())}"
)
print(
    "Working Analysis_Role values : "
    f"{sorted(calibration_df['Analysis_Role'].unique().tolist())}"
)

assert len(calibration_df) == 5760
assert calibration_df["Test_ID"].nunique() == 96
assert calibration_matrix_df["Test_ID"].nunique() == 96
assert validation_matrix_df["Test_ID"].nunique() == 36
assert set(samples_per_test.unique()) == {60}
assert set(calibration_df["Test_Family"].unique()) == {"STATIC"}
assert set(calibration_df["Analysis_Role"].unique()) == {"CALIBRATION"}

calibration_measurement_ids = set(calibration_df["Test_ID"])
calibration_matrix_ids = set(calibration_matrix_df["Test_ID"])
validation_matrix_ids = set(validation_matrix_df["Test_ID"])

assert calibration_measurement_ids == calibration_matrix_ids
assert calibration_measurement_ids.isdisjoint(validation_matrix_ids)

print("\nCM-01 boundary audit: PASS")

## 6. CM-02 — Test_ID-Level Calibration Design

Now the 5,760 repeated samples are reduced to one traceable record per Test_ID.

The expected modeling design is:

- RH setpoints: 20, 40, 60, 80 %RH;
- temperature setpoints: 15, 25, 35 °C;
- Direction: Up / Down;
- Day: 1 / 2;
- Run: 1 / 2;
- 24 unique physical-condition groups defined by RH × Temperature × Direction;
- four Day × Run repetitions per physical-condition group.

This structure will later support group-aware cross-validation without allowing repetitions of the same physical condition to leak across training and assessment folds.

In [ ]:
# ------------------------------------------------------------
# CM-02A: Build one calibration record per Test_ID
# ------------------------------------------------------------

calibration_summary_df = summarize_calibration_test_ids(calibration_df)
design = calibration_design_summary(calibration_summary_df)

print("Calibration design audit")
print("------------------------")
for key, value in design.items():
    print(f"{key:28s}: {value}")

assert len(calibration_summary_df) == 96
assert design["N_Test_IDs"] == 96
assert design["RH_Levels"] == [20.0, 40.0, 60.0, 80.0]
assert design["Temperature_Levels_C"] == [15.0, 25.0, 35.0]
assert {str(x).upper() for x in design["Directions"]} == {"DOWN", "UP"}
assert design["Days"] == [1, 2]
assert design["Runs"] == [1, 2]
assert design["N_Condition_Groups"] == 24
assert design["Samples_Per_Test_ID_Unique"] == [60]

condition_replication = (
    calibration_summary_df
    .groupby("Condition_ID")
    .size()
)

assert set(condition_replication.unique()) == {4}

display(
    calibration_summary_df[
        [
            "Test_ID",
            "RH_Setpoint",
            "Temperature_Setpoint_C",
            "Direction",
            "Day_ID",
            "Run_ID",
            "N_Samples",
            "RH_DUT_Mean",
            "RH_Reference_Mean",
            "Temperature_DUT_Mean_C",
            "Error_RH_Pre",
            "Correction_RH",
            "Condition_ID",
        ]
    ].head(12).round(4)
)

print("\nCM-02 Test_ID summary audit: PASS")

## 7. CM-03 — Calibration Target Verification

Before any model is created, verify the sign convention algebraically and numerically.

For every Test_ID:

```text
Error_RH_Pre + Correction_RH = 0
```

because:

```text
Error_RH_Pre = RH_DUT_Mean - RH_Reference_Mean
Correction_RH = RH_Reference_Mean - RH_DUT_Mean
```

This check prevents a sign error from propagating through the entire calibration workflow.

In [ ]:
# ------------------------------------------------------------
# CM-03A: Verify correction-target sign convention
# ------------------------------------------------------------

correction_identity_error = (
    calibration_summary_df["Correction_RH"]
    - (
        calibration_summary_df["RH_Reference_Mean"]
        - calibration_summary_df["RH_DUT_Mean"]
    )
)

sign_identity_error = (
    calibration_summary_df["Error_RH_Pre"]
    + calibration_summary_df["Correction_RH"]
)

assert np.allclose(correction_identity_error, 0.0)
assert np.allclose(sign_identity_error, 0.0)

print(
    "Maximum correction-definition discrepancy : "
    f"{correction_identity_error.abs().max():.3e} %RH"
)
print(
    "Maximum Error + Correction discrepancy     : "
    f"{sign_identity_error.abs().max():.3e} %RH"
)
print("\nCM-03 correction-target verification: PASS")

## 8. CM-04 — Modeling Table and Predictor-Leakage Audit

The deployable model-selection track starts with only DUT-observable predictors:

```text
RH_DUT_Mean
Temperature_DUT_Mean_C
```

The target is:

```text
Correction_RH
```

The following variables remain available for design control or diagnostics but are **not deployable predictors**:

- `RH_Reference_Mean`;
- `Temperature_Reference_Mean_C`;
- `RH_Setpoint`;
- `Temperature_Setpoint_C`;
- `Day_ID`;
- `Run_ID`;
- `Test_ID`.

`Direction` is retained as a diagnostic design factor because HV-004 showed substantial UP/DOWN structure. It is not automatically eligible for the final deployable model.

In [ ]:
# ------------------------------------------------------------
# CM-04A: Define deployable predictors, target, groups, diagnostics
# ------------------------------------------------------------

deployable_predictors = [
    "RH_DUT_Mean",
    "Temperature_DUT_Mean_C",
]

target_column = "Correction_RH"
group_column = "Condition_ID"

diagnostic_columns = [
    "RH_Setpoint",
    "Temperature_Setpoint_C",
    "Direction",
    "Day_ID",
    "Run_ID",
    "Error_RH_Pre",
]

prohibited_predictor_columns = {
    "RH_Reference_Mean",
    "Temperature_Reference_Mean_C",
    "RH_Setpoint",
    "Temperature_Setpoint_C",
    "Day_ID",
    "Run_ID",
    "Test_ID",
}

assert set(deployable_predictors).isdisjoint(prohibited_predictor_columns)
assert target_column not in deployable_predictors
assert group_column not in deployable_predictors

X_deployable = calibration_summary_df[deployable_predictors].copy()
y_correction = calibration_summary_df[target_column].copy()
cv_groups = calibration_summary_df[group_column].copy()

modeling_table_df = calibration_summary_df[
    [
        "Test_ID",
        group_column,
        *deployable_predictors,
        target_column,
        *diagnostic_columns,
    ]
].copy()

assert len(X_deployable) == 96
assert len(y_correction) == 96
assert len(cv_groups) == 96
assert X_deployable.isna().sum().sum() == 0
assert y_correction.isna().sum() == 0
assert cv_groups.nunique() == 24

print("Deployable predictors :", deployable_predictors)
print("Target                :", target_column)
print("CV grouping variable  :", group_column)
print("Modeling rows         :", len(modeling_table_df))
print("Condition groups      :", cv_groups.nunique())

display(modeling_table_df.head(12).round(4))

print("\nCM-04 predictor-leakage audit: PASS")

## 9. Initial Curated Evidence Export

At this stage only the calibration-design summary is exported.

No model-comparison table, fitted coefficients, or validation result exists yet.

In [ ]:
# ------------------------------------------------------------
# Export initial HV-006 design summary
# ------------------------------------------------------------

design_summary_export = pd.DataFrame(
    [
        {"Item": "Measurement rows", "Value": len(calibration_df)},
        {"Item": "Calibration Test_IDs", "Value": design["N_Test_IDs"]},
        {"Item": "RH levels", "Value": ", ".join(map(str, design["RH_Levels"]))},
        {
            "Item": "Temperature levels C",
            "Value": ", ".join(map(str, design["Temperature_Levels_C"])),
        },
        {"Item": "Directions", "Value": ", ".join(design["Directions"])},
        {"Item": "Days", "Value": ", ".join(map(str, design["Days"]))},
        {"Item": "Runs", "Value": ", ".join(map(str, design["Runs"]))},
        {
            "Item": "Condition groups",
            "Value": design["N_Condition_Groups"],
        },
        {
            "Item": "Samples per Test_ID",
            "Value": ", ".join(
                map(str, design["Samples_Per_Test_ID_Unique"])
            ),
        },
        {
            "Item": "Reserved validation Test_IDs",
            "Value": validation_matrix_df["Test_ID"].nunique(),
        },
    ]
)

design_summary_path = (
    tables_dir / "hv006_calibration_design_summary.csv"
)

design_summary_export.to_csv(design_summary_path, index=False)

display(design_summary_export)
print(f"\nExported: {design_summary_path}")

## 10. Initial HV-006 Checkpoint

The CALIBRATION-only boundary, Test_ID summary, correction-target sign convention, and predictor-leakage checks have been implemented.

The required clean-kernel checkpoint is:

```text
CM-01 boundary audit: PASS
CM-02 Test_ID summary audit: PASS
CM-03 correction-target verification: PASS
CM-04 predictor-leakage audit: PASS
```

Once those four checks pass, the notebook may proceed to the baseline model stage below.

The next question is deliberately modest:

> How much improvement is obtained by doing nothing (C0) versus applying only one global offset (C1), before introducing RH- or temperature-dependent calibration structure?


## 11. Why C0 and C1 Come First

Before fitting slope, temperature, interaction, or polynomial terms, two baselines are required.

### C0 — Uncalibrated baseline

```text
Predicted_Correction_RH = 0
```

This reproduces the raw Test_ID-level DUT performance and establishes the error level that calibration must beat.

### C1 — Constant-offset calibration

```text
Predicted_Correction_RH = mean(training Correction_RH)
```

This asks whether one global additive correction is sufficient.

The C1 correction for each assessment fold must be estimated from **training rows only**. Using the overall mean correction before cross-validation would leak information from the assessment fold into the fitted offset.

HV-004 already indicated that a constant offset is unlikely to explain the full error structure, but C1 remains essential as a quantitative baseline.

## 12. CM-05 — Frozen Group-Aware Cross-Validation Structure

The primary internal comparison uses `GroupKFold` with:

```text
group = Condition_ID
Condition_ID = RH_Setpoint × Temperature_Setpoint_C × Direction
```

There are 24 physical-condition groups and four Day × Run Test_ID repetitions inside each group.

The fold rule is:

> all repetitions of one physical condition must remain together in either training or assessment data for a fold.

This prevents the model from being assessed on a condition whose near-replicates were used during fitting.

A six-fold split is used here:

```text
24 condition groups / 6 folds = 4 held-out condition groups per fold
4 Test_ID repetitions/group   = 16 held-out Test_IDs per fold
```

The same frozen split list will be reused for later C2-C4 comparisons so model rankings are made on identical assessment partitions.

In [ ]:
# ------------------------------------------------------------
# CM-05A: Freeze and audit grouped cross-validation partitions
# ------------------------------------------------------------

n_cv_splits = 6
group_cv = GroupKFold(n_splits=n_cv_splits)

cv_splits = list(
    group_cv.split(
        X_deployable,
        y_correction,
        groups=cv_groups,
    )
)

fold_audit_records = []
all_test_indices = []

for fold_id, (train_idx, test_idx) in enumerate(cv_splits, start=1):
    train_groups = set(cv_groups.iloc[train_idx])
    test_groups = set(cv_groups.iloc[test_idx])

    assert train_groups.isdisjoint(test_groups)

    all_test_indices.extend(test_idx.tolist())

    fold_audit_records.append(
        {
            "Fold": fold_id,
            "N_Train_Rows": len(train_idx),
            "N_Test_Rows": len(test_idx),
            "N_Train_Groups": len(train_groups),
            "N_Test_Groups": len(test_groups),
            "Test_RH_Levels": ", ".join(
                map(
                    lambda x: f"{x:g}",
                    sorted(
                        calibration_summary_df.iloc[test_idx][
                            "RH_Setpoint"
                        ].unique()
                    ),
                )
            ),
            "Test_Temperature_Levels_C": ", ".join(
                map(
                    lambda x: f"{x:g}",
                    sorted(
                        calibration_summary_df.iloc[test_idx][
                            "Temperature_Setpoint_C"
                        ].unique()
                    ),
                )
            ),
            "Test_Directions": ", ".join(
                sorted(
                    calibration_summary_df.iloc[test_idx][
                        "Direction"
                    ].astype(str).unique()
                )
            ),
        }
    )

fold_audit_df = pd.DataFrame(fold_audit_records)

assert len(cv_splits) == 6
assert sorted(all_test_indices) == list(range(len(calibration_summary_df)))
assert len(set(all_test_indices)) == len(calibration_summary_df)
assert set(fold_audit_df["N_Test_Rows"]) == {16}
assert set(fold_audit_df["N_Test_Groups"]) == {4}

display(fold_audit_df)

print("\nCM-05 grouped cross-validation audit: PASS")

## 13. Calibration Residual Metrics

For every candidate model, performance is evaluated from **out-of-fold calibrated predictions**.

The post-calibration residual is:

```text
Residual_Cal = RH_Calibrated - RH_Reference
```

The primary comparison metric is RMSE.

The notebook also reports:

- MBE — residual center / signed bias;
- MAE — average absolute residual magnitude;
- median residual;
- residual standard deviation;
- maximum absolute residual;
- R² for calibrated RH versus reference RH.

R² is descriptive only. Because this dataset spans a broad RH range, a high R² can coexist with practically important calibration error. It is therefore never used alone to select a model.

In [ ]:
# ------------------------------------------------------------
# Reusable metric helper for out-of-fold calibration results
# ------------------------------------------------------------

def calibration_metrics(
    residual: pd.Series | np.ndarray,
    rh_reference: pd.Series | np.ndarray,
    rh_calibrated: pd.Series | np.ndarray,
) -> dict[str, float]:
    residual_array = np.asarray(residual, dtype=float)
    reference_array = np.asarray(rh_reference, dtype=float)
    calibrated_array = np.asarray(rh_calibrated, dtype=float)

    return {
        "N": int(residual_array.size),
        "MBE": float(np.mean(residual_array)),
        "MAE": float(np.mean(np.abs(residual_array))),
        "RMSE": float(np.sqrt(np.mean(residual_array**2))),
        "Median_Error": float(np.median(residual_array)),
        "Residual_SD": float(np.std(residual_array, ddof=1)),
        "Max_Abs_Error": float(np.max(np.abs(residual_array))),
        "R2": float(r2_score(reference_array, calibrated_array)),
    }

## 14. C0 — Uncalibrated Baseline

C0 applies no correction:

```text
Predicted_Correction_C0 = 0
RH_Calibrated_C0        = RH_DUT_Mean
Residual_C0             = Error_RH_Pre
```

C0 is not a fitted calibration model. It is the baseline that all later candidates must improve upon.

In [ ]:
# ------------------------------------------------------------
# C0: Uncalibrated out-of-fold baseline
# ------------------------------------------------------------

baseline_results_df = calibration_summary_df[
    [
        "Test_ID",
        "Condition_ID",
        "RH_Setpoint",
        "Temperature_Setpoint_C",
        "Direction",
        "Day_ID",
        "Run_ID",
        "RH_DUT_Mean",
        "RH_Reference_Mean",
        "Error_RH_Pre",
        "Correction_RH",
    ]
].copy()

baseline_results_df["Predicted_Correction_C0"] = 0.0
baseline_results_df["RH_Calibrated_C0"] = baseline_results_df["RH_DUT_Mean"]
baseline_results_df["Residual_C0"] = (
    baseline_results_df["RH_Calibrated_C0"]
    - baseline_results_df["RH_Reference_Mean"]
)

assert np.allclose(
    baseline_results_df["Residual_C0"],
    baseline_results_df["Error_RH_Pre"],
)

c0_metrics = calibration_metrics(
    residual=baseline_results_df["Residual_C0"],
    rh_reference=baseline_results_df["RH_Reference_Mean"],
    rh_calibrated=baseline_results_df["RH_Calibrated_C0"],
)

display(pd.DataFrame([{"Model": "C0_Uncalibrated", **c0_metrics}]).round(4))

## 15. C1 — Cross-Validated Constant-Offset Calibration

For each fold, C1 estimates one correction from the training portion only:

```text
Offset_fold = mean(Correction_RH in training rows)
```

That frozen fold-specific offset is then applied to every Test_ID in the held-out fold.

This provides an honest out-of-fold estimate of what a single global offset can accomplish without allowing assessment information to influence the offset.

In [ ]:
# ------------------------------------------------------------
# C1: Out-of-fold constant-offset calibration
# ------------------------------------------------------------

c1_predicted_correction = np.full(
    len(calibration_summary_df),
    np.nan,
    dtype=float,
)

c1_fold_records = []

for fold_id, (train_idx, test_idx) in enumerate(cv_splits, start=1):
    training_correction = y_correction.iloc[train_idx]
    fold_offset = float(training_correction.mean())

    c1_predicted_correction[test_idx] = fold_offset

    c1_fold_records.append(
        {
            "Fold": fold_id,
            "Train_Mean_Correction_RH": fold_offset,
            "N_Train": len(train_idx),
            "N_Test": len(test_idx),
        }
    )

assert not np.isnan(c1_predicted_correction).any()

baseline_results_df["Predicted_Correction_C1"] = c1_predicted_correction
baseline_results_df["RH_Calibrated_C1"] = (
    baseline_results_df["RH_DUT_Mean"]
    + baseline_results_df["Predicted_Correction_C1"]
)
baseline_results_df["Residual_C1"] = (
    baseline_results_df["RH_Calibrated_C1"]
    - baseline_results_df["RH_Reference_Mean"]
)

c1_metrics = calibration_metrics(
    residual=baseline_results_df["Residual_C1"],
    rh_reference=baseline_results_df["RH_Reference_Mean"],
    rh_calibrated=baseline_results_df["RH_Calibrated_C1"],
)

c1_fold_df = pd.DataFrame(c1_fold_records)

display(c1_fold_df.round(4))
display(pd.DataFrame([{"Model": "C1_Constant_Offset", **c1_metrics}]).round(4))

## 16. C0 vs C1 — Baseline Comparison

The first calibration question is now answerable:

> Does one global additive correction materially improve the raw sensor result?

The comparison below uses the same 96 Test_ID records and reports C1 only from out-of-fold predictions.

In [ ]:
# ------------------------------------------------------------
# Compare C0 and C1 using out-of-fold results
# ------------------------------------------------------------

baseline_comparison_df = pd.DataFrame(
    [
        {"Model": "C0_Uncalibrated", **c0_metrics},
        {"Model": "C1_Constant_Offset", **c1_metrics},
    ]
)

baseline_comparison_df["RMSE_Improvement_vs_C0_pct"] = (
    100.0
    * (
        c0_metrics["RMSE"]
        - baseline_comparison_df["RMSE"]
    )
    / c0_metrics["RMSE"]
)

baseline_comparison_df["MAE_Improvement_vs_C0_pct"] = (
    100.0
    * (
        c0_metrics["MAE"]
        - baseline_comparison_df["MAE"]
    )
    / c0_metrics["MAE"]
)

display(
    baseline_comparison_df[
        [
            "Model",
            "N",
            "MBE",
            "MAE",
            "RMSE",
            "Residual_SD",
            "Max_Abs_Error",
            "R2",
            "RMSE_Improvement_vs_C0_pct",
            "MAE_Improvement_vs_C0_pct",
        ]
    ].round(4)
)

## 17. Baseline Residual Structure Check

A lower overall RMSE does not prove that C1 is an adequate calibration.

The residual means are therefore summarized by RH, temperature, and Direction.

If C1 removes only the global center while substantial condition-dependent residual structure remains, that is evidence that a more informative model such as C2 or C3 is justified.

In [ ]:
# ------------------------------------------------------------
# Condition-level residual structure for C0 and C1
# ------------------------------------------------------------

residual_by_rh_df = (
    baseline_results_df
    .groupby("RH_Setpoint", as_index=False)
    .agg(
        N=("Test_ID", "size"),
        C0_MBE=("Residual_C0", "mean"),
        C1_MBE=("Residual_C1", "mean"),
        C0_MAE=("Residual_C0", lambda x: x.abs().mean()),
        C1_MAE=("Residual_C1", lambda x: x.abs().mean()),
    )
)

residual_by_temperature_df = (
    baseline_results_df
    .groupby("Temperature_Setpoint_C", as_index=False)
    .agg(
        N=("Test_ID", "size"),
        C0_MBE=("Residual_C0", "mean"),
        C1_MBE=("Residual_C1", "mean"),
        C0_MAE=("Residual_C0", lambda x: x.abs().mean()),
        C1_MAE=("Residual_C1", lambda x: x.abs().mean()),
    )
)

residual_by_direction_df = (
    baseline_results_df
    .groupby("Direction", as_index=False)
    .agg(
        N=("Test_ID", "size"),
        C0_MBE=("Residual_C0", "mean"),
        C1_MBE=("Residual_C1", "mean"),
        C0_MAE=("Residual_C0", lambda x: x.abs().mean()),
        C1_MAE=("Residual_C1", lambda x: x.abs().mean()),
    )
)

print("Residual structure by RH setpoint")
display(residual_by_rh_df.round(4))

print("Residual structure by temperature")
display(residual_by_temperature_df.round(4))

print("Residual structure by Direction")
display(residual_by_direction_df.round(4))

## 18. Baseline-Model Checkpoint

Before C2 is introduced, verify that:

- `CM-05 grouped cross-validation audit: PASS` is reported;
- every Test_ID receives exactly one out-of-fold assessment prediction;
- C0 reproduces the original pre-calibration residual exactly;
- C1 estimates its offset from training rows only;
- C0 and C1 metrics are computed from Test_ID-level results;
- the residual summaries by RH, temperature, and Direction are visible.

### Interpretation rule

Do **not** choose C1 merely because its MBE is closer to zero.

A constant offset can center the residual distribution while leaving strong RH-, temperature-, or Direction-dependent structure.

The next stage, if these checks pass, is:

```text
C2 — RH-only linear correction
C3 — RH + DUT-temperature linear correction
```

Those models will reuse the exact same frozen `cv_splits` defined in CM-05.